In [30]:
import pandas as pd
import numpy as np

## Merge Contract with Weighted Season and Playoff Stats

We combine the contract data with weighted regular season and playoff statistics.
For each contract year, we compute a weighted average of the stats from the two prior seasons and the contract season:
- weight (year-2): 0.2
- weight (year-1): 0.3
- weight (year):   0.5

The resulting dataframe includes:
- Player, YRS, is_retained, year (contract year)
- age (from regular season stats for the contract year)
- Weighted regular season stats (suffixed `_reg`)
- Weighted playoff stats (suffixed `_playoff`)


In [52]:
# Load data
contract_df = pd.read_csv('../../data/processed/contract_data.csv')
stats_reg_df = pd.read_csv('../../data/processed/Stats_reg_2014_2024_cleaned.csv')
stats_playoff_df = pd.read_csv('../../data/processed/Stats_playoffs_2014_2024_cleaned.csv')
health_df = pd.read_csv('../../data/processed/health.csv')

# Identify numeric statistic columns (exclude identifier columns)
id_cols = ['Player', 'year', 'Team', 'Pos', 'Age']  # Age will be used separately
stat_cols_reg = [c for c in stats_reg_df.columns if c not in id_cols]
stat_cols_playoff = [c for c in stats_playoff_df.columns if c not in id_cols]
stat_cols_health = ['MIN_PER_GAME', 'ATTENDANCE_RATE', 'MAJOR_INJURY'] # 指定健康數據需要加權的欄位

# Helper to compute weighted average for a given player and target year
# Helper to compute dynamic weighted average (支援 1~3 年動態加權)
def weighted_stats(df, player, target_year, stat_cols, weights=[0.2, 0.3, 0.5]):
    """
    Return dynamically weighted average of stats for player across target_year-2, target_year-1, target_year.
    If a year is missing, the weights are re-normalized among the available years.
    """
    # 篩選出數值型欄位
    numeric_stat_cols = df[stat_cols].select_dtypes(include=[np.number]).columns
    years_needed = [target_year - 2, target_year - 1, target_year]
    
    available_data = []
    available_weights = []
    
    # 逐年檢查是否有資料
    for w, y in zip(weights, years_needed):
        sub = df[(df['Player'] == player) & (df['year'] == y)]
        if not sub.empty:
            available_data.append(sub[numeric_stat_cols].iloc[0])
            available_weights.append(w)
            
    # 如果這三年「完全沒有任何一筆資料」，才回傳 NaN
    if not available_data:
        return pd.Series([np.nan] * len(numeric_stat_cols), index=numeric_stat_cols)
        
    # 重頭戲：重新標準化權重 (讓總和等於 1)
    total_weight = sum(available_weights)
    normalized_weights = [w / total_weight for w in available_weights]
    
    # 計算重新加權後的平均值
    weighted = pd.Series(0.0, index=numeric_stat_cols)
    for w, data in zip(normalized_weights, available_data):
        weighted += w * data
        
    return weighted

# Prepare list to hold enriched contract rows
enriched_rows = []

for _, row in contract_df.iterrows():
    player = row['Player']
    yr = int(row['year'])  # contract year
    yrs = row['YRS']
    retained = row['is_retained']
    cap_pct = row['Cap_Pct'] 

    # 1. Get Age, Team, and Pos from regular season stats for the contract year
    # [新增] 把 Pos 也一併抓出來
    info_sub = stats_reg_df[(stats_reg_df['Player'] == player) & (stats_reg_df['year'] == yr)]
    if not info_sub.empty:
        age = info_sub['Age'].iloc[0]
        team = info_sub['Team'].iloc[0]
        pos = info_sub['Pos'].iloc[0] 
    else:
        age = np.nan
        team = 'Unknown'
        pos = 'Unknown'                

    # 2. Compute weighted stats (例行賽、季後賽、健康數據)
    wt_reg = weighted_stats(stats_reg_df, player, yr, stat_cols_reg)
    wt_playoff = weighted_stats(stats_playoff_df, player, yr, stat_cols_playoff)
    wt_health = weighted_stats(health_df, player, yr, stat_cols_health)

    # 處理季後賽遺失值與新增經驗旗標
    if wt_playoff.isna().all():
        has_playoff_exp = 0  # 沒打季後賽，設為 0
        wt_playoff = wt_reg.copy()  # 用例行賽數據直接插補
    else:
        has_playoff_exp = 1  # 有打季後賽，設為 1

    # 插補完成後，分別加上後綴區隔
    wt_reg = wt_reg.add_suffix('_reg')
    wt_playoff = wt_playoff.add_suffix('_playoff')
    wt_health = wt_health.add_suffix('_health')

    # Build row 
    new_row = {
        'Player': player,
        'Team': team,         
        'Pos': pos,           
        'YRS': yrs,
        'is_retained': retained,
        'year': yr,
        'age': age,
        'has_playoff_exp': has_playoff_exp,
        'Cap_Pct': cap_pct
    }
    
    # Add weighted stats to the row
    for col in wt_reg.index:
        new_row[col] = wt_reg[col]
    for col in wt_playoff.index:
        new_row[col] = wt_playoff[col]
    for col in wt_health.index:
        new_row[col] = wt_health[col]

    enriched_rows.append(new_row)

# Create final dataframe
merged_df = pd.DataFrame(enriched_rows)

cols = ['Player', 'Team', 'Pos', 'year', 'YRS', 'is_retained', 'age', 'has_playoff_exp', 'Cap_Pct'] \
        + sorted([c for c in merged_df.columns if c.endswith('_reg')]) \
        + sorted([c for c in merged_df.columns if c.endswith('_playoff')]) \
        + sorted([c for c in merged_df.columns if c.endswith('_health')])

merged_df = merged_df[cols]
print(merged_df.shape)
merged_df.head()

(977, 100)


,Player,Team,Pos,year,YRS,is_retained,age,has_playoff_exp,Cap_Pct,2P%_reg,...,TRB_playoff,TS%_playoff,USG%_playoff,VORP_playoff,WS/48_playoff,WS_playoff,eFG%_playoff,ATTENDANCE_RATE_health,MAJOR_INJURY_health,MIN_PER_GAME_health
0,stephen curry,GSW,PG,2017,5,1,28.0,1,0.3500,0.5439,...,5.750000,0.631800,30.830000,1.760000,0.227200,3.050000,0.579700,0.965854,0.0,33.485785
1,blake griffin,LAC,PF,2017,5,1,27.0,1,0.3000,0.5090,...,8.180000,0.534100,27.980000,0.370000,0.104000,0.580000,0.483200,0.663415,0.8,34.081031
2,gordon hayward,UTA,SF,2017,4,0,26.0,1,0.3000,0.4923,...,6.100000,0.598000,28.500000,0.800000,0.142000,1.200000,0.516000,0.923171,0.2,34.966463
3,jrue holiday,NOP,PG,2017,5,1,26.0,1,0.2592,0.4862,...,1.000000,0.458000,21.400000,0.000000,-0.028000,0.000000,0.395000,0.743902,0.3,31.307802
4,otto porter,WAS,SF,2017,4,0,23.0,1,0.2500,0.5470,...,7.214286,0.600429,13.585714,0.514286,0.159714,1.357143,0.562429,0.942683,0.0,29.255610


In [57]:
merged_df = merged_df[cols]
merged_df.fillna(0, inplace=True)

# 只要是 Unknown 的，代表他們缺乏前兩年的 NBA 實戰數據，直接刪除不納入模型訓練
merged_df = merged_df[merged_df['Team'] != 'Unknown'].reset_index(drop=True)

print("✅ 最終乾淨資料維度:", merged_df.shape)
merged_df.head()

✅ 最終乾淨資料維度: (898, 100)


,Player,Team,Pos,year,YRS,is_retained,age,has_playoff_exp,Cap_Pct,2P%_reg,...,TRB_playoff,TS%_playoff,USG%_playoff,VORP_playoff,WS/48_playoff,WS_playoff,eFG%_playoff,ATTENDANCE_RATE_health,MAJOR_INJURY_health,MIN_PER_GAME_health
0,stephen curry,GSW,PG,2017,5,1,28.0,1,0.3500,0.5439,...,5.750000,0.631800,30.830000,1.760000,0.227200,3.050000,0.579700,0.965854,0.0,33.485785
1,blake griffin,LAC,PF,2017,5,1,27.0,1,0.3000,0.5090,...,8.180000,0.534100,27.980000,0.370000,0.104000,0.580000,0.483200,0.663415,0.8,34.081031
2,gordon hayward,UTA,SF,2017,4,0,26.0,1,0.3000,0.4923,...,6.100000,0.598000,28.500000,0.800000,0.142000,1.200000,0.516000,0.923171,0.2,34.966463
3,jrue holiday,NOP,PG,2017,5,1,26.0,1,0.2592,0.4862,...,1.000000,0.458000,21.400000,0.000000,-0.028000,0.000000,0.395000,0.743902,0.3,31.307802
4,otto porter,WAS,SF,2017,4,0,23.0,1,0.2500,0.5470,...,7.214286,0.600429,13.585714,0.514286,0.159714,1.357143,0.562429,0.942683,0.0,29.255610


### Merge 球隊與球員數據

In [58]:
team_salary_cap_df = pd.read_csv('../../data/external/team_salary_cap.csv')
team_salary_cap_df

,Team,Total Cap Allocations,Cap Space All,year,Salary_Cap,Payroll_Pct,Cap_Space_Pct
0,DAL,85147033.0,"$13,945,967",2017,94143000.0,0.904444,0.095556
1,CHI,90105625.0,"$8,987,375",2017,94143000.0,0.957114,0.042886
2,PHO,92518634.0,"$6,574,366",2017,94143000.0,0.982746,0.017254
3,IND,93661969.0,"$5,431,031",2017,94143000.0,0.994890,0.005110
4,ORL,95538311.0,"$3,554,689",2017,94143000.0,1.014821,0.000000
...,...,...,...,...,...,...,...
235,GSW,199518342.0,"$-58,930,342",2024,136021000.0,1.466820,0.000000
236,LAL,200785985.0,"$-60,197,985",2024,136021000.0,1.476140,0.000000
237,WAS,211677970.0,"$-71,089,970",2024,136021000.0,1.556215,0.000000
238,PHO,228464502.0,"$-87,876,502",2024,136021000.0,1.679627,0.000000


In [59]:
cols_to_merge = ['Team', 'year', 'Payroll_Pct', 'Cap_Space_Pct']

final_df = pd.merge(merged_df, team_salary_cap_df[cols_to_merge], on=['Team', 'year'], how='left')

final_df

,Player,Team,Pos,year,YRS,is_retained,age,has_playoff_exp,Cap_Pct,2P%_reg,...,USG%_playoff,VORP_playoff,WS/48_playoff,WS_playoff,eFG%_playoff,ATTENDANCE_RATE_health,MAJOR_INJURY_health,MIN_PER_GAME_health,Payroll_Pct,Cap_Space_Pct
0,stephen curry,GSW,PG,2017,5,1,28.0,1,0.3500,0.5439,...,30.830000,1.760000,0.227200,3.050000,0.579700,0.965854,0.0,33.485785,1.453567,0.0
1,blake griffin,LAC,PF,2017,5,1,27.0,1,0.3000,0.5090,...,27.980000,0.370000,0.104000,0.580000,0.483200,0.663415,0.8,34.081031,1.262091,0.0
2,gordon hayward,UTA,SF,2017,4,0,26.0,1,0.3000,0.4923,...,28.500000,0.800000,0.142000,1.200000,0.516000,0.923171,0.2,34.966463,1.135106,0.0
3,jrue holiday,NOP,PG,2017,5,1,26.0,1,0.2592,0.4862,...,21.400000,0.000000,-0.028000,0.000000,0.395000,0.743902,0.3,31.307802,1.277681,0.0
4,otto porter,WAS,SF,2017,4,0,23.0,1,0.2500,0.5470,...,13.585714,0.514286,0.159714,1.357143,0.562429,0.942683,0.0,29.255610,1.318357,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
893,kelly olynyk,TOR,C,2024,2,1,32.0,0,0.0911,0.5942,...,18.390000,1.350000,0.119100,4.060000,0.586300,0.821951,0.0,23.658347,1.312268,0.0
894,richaun holmes,WAS,C,2024,2,1,30.0,0,0.0900,0.5994,...,14.620000,-0.050000,0.119900,1.530000,0.606900,0.507317,0.0,14.182735,1.556215,0.0
895,mike conley,MIN,PG,2024,2,1,36.0,1,0.0710,0.4791,...,16.700000,0.310000,0.088700,0.750000,0.519400,0.884146,0.0,29.228541,1.740363,0.0
896,john konchar,MEM,SF,2024,3,1,27.0,1,0.0439,0.5695,...,8.860000,0.000000,0.047200,0.040000,0.260400,0.774390,0.0,20.480649,1.233743,0.0


In [62]:
final_df.dropna().to_csv('../../data/processed/merged_weighted_stats.csv', index=False)

完成訓練集彙整